In [11]:
#===============================
#Smart expense tracker
#==============================

In [12]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 


In [13]:
#Constants used by the application
#The required sample dataset is named expense.csv.
CSV_FILE = "expense.csv"

#only these categories are accepted when a new expense is entered.
VALID_CATEGORIES=[
    "Food",
    "Transport",
    "Utilities",
    "Entertainment",
    "Healthcare",
    "Shopping",
    "Education",
    "Other"
]

#these are the coulumn names used in the csv file and dataFrame
COLUMNS = ["date", "amount", "category", "Description"]


In [ ]:
#ExpenseTracker class(Object-Oriented Programming)
class ExpenseTracker:
    #the constructor loads the existing CSV file or creates an empty dataset.
    def __init__(self, filename=CSV_FILE):
        self.filename=filename
        self.expenses=pd.DataFrame(columns=COLUMNS)
        self.load_data()

    #load and clean the expense dataset using pandas
    def load_data(self):
        #create a new csv file if it does not already exists
        if not os.path.exists(self.filename):
            self.expenses=pd.DataFrame(columns=COLUMNS)
            self.save_data()
            return
        try:
            #read the existing csv file into a pandas dataframe
            self.expenses = pd.read_csv(self.filename)

            #ensure all required coulumns exists
            for column in COLUMNS:
                if column not in self.expenses.columns:
                    self.expenses[column] = ""

            #keep the required columns and their order
            self.expenses = self.expenses[COLUMNS]

            #Clean invalid and missing data
            self.clean_data()

        except(pd.errors.EmptyDataError, pd.errors.ParserError, OSError) as error:
            print("The CSV file could not be read:", error)
            print("Starting with an empty expense dataset")
            self.expenses = pd.DataFrame(columns=COLUMNS)

    def clean_data(self):
        #convert dates to real dates, invalid dates become missing values
        self.expenses["date"] = pd.to_datetime(
            self.expenses["date"], format="%Y-%m-%d", errors="coerce"
        )

        #convert amounts to number. Text or invalid amounts become missing value
        self.expenses["amount"] = pd.to_numeric(
            self.expenses["amount"], errors="coerce"
        )

        #remove extra spaces from category and description text
        self.expenses["category"] = self.expenses["category"].fillna("").astype(str).str.strip()
        self.expenses["Description"]=self.expenses["Description"].fillna("").astype(str).str.strip()

        #Standardize category spelling and capitalization
        self.expenses["category"]=self.expenses["category"].str.title()

        #remove rows with missing/invalid dates, amount, categories or descriptions
        self.expenses.dropna(subset=["date","amount"], inplace=True)
        self.expenses = self.expenses[
            (self.expenses["amount"]>0)
            & (self.expenses["category"] != "")
            & (self.expenses["Description"] != "")
            ]

        #remove categories that are not in the application's valid category list
        self.expenses = self.expenses[
            self.expenses["category"].isin(VALID_CATEGORIES)
            ]

        #reset row numbering after removing invalid records
        self.expenses.reset_index(drop=True, inplace=True)

        #Save the cleaned dataset back to the csv file
        self.save_data()

    #Save the current dataframe to the csv file
    def save_data(self):
        #stores dates
        data_to_save = self.expenses.copy()

        if not data_to_save.empty:
            data_to_save["date"]=pd.to_datetime(
                data_to_save["date"], errors="coerce"
            ).dt.strftime("%Y-%m-%d")

        data_to_save.to_csv(self.filename, index=False)

#add a new expense after validating the input
    def add_expenses(self, date, amount, category, description):
        try:
            valid_date = datetime.strptime(date.strip(), "%Y-%m-%d").date()
        except (ValueError, AttributeError):
            print("Invalid date, use YYYY-MM-DD")
            return False

        try:
            valid_amount = float(amount)
        except (ValueError, TypeError):
            print("invalid amount")
            return False

        if valid_amount <= 0:
            print("amount must be greater")
            return False

        #valid the category
        valid_category = str(category).strip().title()
        if valid_category not in VALID_CATEGORIES:
            print("invalid category")
            return False

        #validate the description
        valid_description = str(description).strip()
        if not valid_description:
            print("Description invalid")
            return False

        #create a new row and add it to the DataFrame
        new_expense = pd.DataFrame([{
            "date" : pd.Timestamp(valid_date),
            "amount" : valid_amount,
            "category" : valid_category,
            "Description" : valid_description
        }])
        self.expenses = pd.concat(
            [self.expenses, new_expenses], ignore_index=True)

        #save
        self.save_data()
        print("expense added")
        return True

        #calculate
    def get_summary(self):
        amounts = np.array(self.expenses["amount"],dtype=float)

        total_expenses = float(np.sum(amounts)) if len(amounts) > 0 else 0.0
        average_expenses = float(np.mean(amounts)) if len(amounts) > 0 else 0.0

        #pandas
        category_totals = self.expenses.groupby("category")["amount"].sum().sort_values(
               ascending=False
        )

        #calculate the percantage
        if total_expenses > 0:
            category_percentages=(category_totals/total_expenses) * 100
        else:
            category_percentages = category_totals.copy()

        #groupby 
        if not self.expenses.empty:
            monthly_totals=self.expenses.assign(
                month=pd.to_datetime(self.expenses["date"]).dt.to_period("M").astype(str)).groupby("month")["amount"].sum()
            
            monthly_average_spending = float(np.mean
               (np.array(monthly_totals,dtype=float)
               ))if len(monthly_totals)>0 else 0.0
        else:
            monthly_totals = pd.Series(dtype=float)
            monthly_average_spending = 0.0

        return {
            "total" : total_expenses,
            "average" : average_expenses,
            "category_totals" : category_totals,
            "category_percentages" : category_percentages,
            "monthly_totals" : monthly_totals,
            "monthly_average": monthly_average_spending,
            "count" : len(self.expenses)
        }
     #filter expenses by category, date, range or amount   
    def filter_expenses(self,condition):
        filtered = self.expenses.copy()
        category=condition.get("category")
        if category:
            filtered = filtered[
                filtered["category"] == str(category).strip().title()
            ]

        start_date=condition.get("start_date")
        if start_date:
            filtered = filtered[
                pd.to_datetime(filtered["date"]) >= pd.timestamp(start_date)
            ]

        end_date = condition.get("end_date")
        if end_date:
            filtered=filtered[
                pd.to_datetime(filtered["date"]) <= pd.Timestamp(end_date)
            ]

         #filter by minimum amount
        min_amount = condition.get("min_amount")
        if min_amount is not None:
            filtered = filtered[filtered["amount"] >= float(min_amount)]

         #maximum
        max_amount = condition.get("max_amount")
        if max_amount is not None:
            filtered = filtered[filtered["amount"] <= float(max_amount)]

        return filtered.reset_index(drop=True)

    def genrate_reports(self):
        summary = self.get_summary()

        print("========= Expense summary report===")
        print("Number is expenses: ", summary["count"])
        print("Total expenses: {:,2f}".format(summary["total"]))
        print("Average expenses: {:,2f}".format(summary["average"]))
        print("Mohtly average spending: {:,2f}".format(summary["monthly_average"]))

        print("Category-wise spending")
        if summary["category_totals"].empty:
            print("no expense records available")
        else:
            for category, total in summary["category_percentages"].items():
                percentage = summary["category_pacentages"].get(category,0)
                print("{}: {:,2f} ({:,2f}%)".format(
                    category,total,percentage
                ))

        print("Monthly spending")
        if summary["monthly_totals"].empty:
            print("no monthly data availabe")
        else:
            for month, total in summary["monthly_totals"].iotems():
                print("{}: {:,2f}".format(month, total))

        print("================")
        
        #create the four required
    def generate_visualizations(self):
        if self.expenses.empty:
            print("No expense data")
            return
        plot_data = self.expenses.copy()
        
        plot_data["date"]=pd.to_datetime(plot_data["date"], errors="coerce")

        #remove
        plot_data=plot_data.dropna(subset=["date","amount"])

        #stop
        if plot_data.empty:
            print("no valid expenses date is available")
            return
            
        #create a month coulumn
        plot_data["month"]=(
            plot_data["date"].dt.to_periods("M").astype(str)
        )

        #apply
        sns.set_theme(style="whitegrid")

        #create a dashboard with four charts
        figure, axes = plt.subplots(2,2, figsize=(15,10))

        #add a main title
        figure.subtitle(
            "smart expense tracker-spending anaylsis",
            fontsize=16
        )

        #chart1: bar chart
        category_totals=(
            plot_data.groupby("category", as_index=False)["amount"].sum())

        #draw
        sns.barplot(
            data=category_totals,
            x="category",
            y="amount",
            hue="category",
            legend=False,
            ax=axes[0,0]
        )

        axes[0,0].set_title("Total expenses by category")
        axes[0,0].set_xlabel("category")
        axes[0,0].set_ylabel("Total  amount")
        axes[0,0].tick_params(axis="x", rotation=30)

        #chart 2 
        #calculate
        monthly_totals=(
            plot.data.groupby("month", as_index=False)["amount"].sum())

        sns.lineplot(
            data=monthy_totals,
            x="month",
            y="amount",
            marker="o",
            label="Monthly total",
            ax=axes[0,1]
        )

        #add title
        axes[0,1].set_title("spending trends over time")
        axes[0,0].set_xlabel("month")
        axes[0,0].set_ylabel("Total  amount")
        axes[0,0].tick_params(axis="x", rotation=30)
        axes[0,0].legend(title="spending")

        #chart 3 pie chart
        axes[1,0].pie(
            category_totals["amount"],
            labels=category_totals["category"],
            autopct="%1.1f%%",
            startangle=90
        )

        #add title and category legend 
        axes[1,0].set_title("Sepnding Distribution by categpry")
        axes[1,0].legend(
            category_totals["category"],
            title="category",
            loc="best",
            fontsize="small"
        )

        #4.Histogram
        axes[1,1].hist(
            plot_data["amount"],
            bins=10,
            label="expense frequency",
            edgecolor="black"
        )

        #Add tile
        axes[1,1].set_title("Frequency of expenses amounts")
        axes[1,1].set_xlabel("expense amount")
        axes[1,1].set_ylabel("frequency")
        axes[1,1].legends(title="Amounts")

        #adjusting spacing to labels and titles are visible
        plt.tight_layout(rect=[0,0,1,0.96])
        plt.show()

     def input_new_expense(tracker):
         date=input("Enter date(YYYY-MM-DD): ").strip()
         amount=input("Enter amount: ").strip()
         

        
        
        
        

        

        
        
        
                              
                              
        
    
        
        
            
        
        
        
                

        